# Grammar Scoring Engine for Spoken Audio
### Modular Multi-View Pipeline for Continuous Grammar Scoring (0.0 - 5.0)
**Audience / Author**: Coding Agent (SHL Hiring Assessment Deliverable)  
**Deliverable**: `grammar_scoring.ipynb` (Full Pipeline Notebook) + `src/` Package + `outputs/submission.csv`

---

### Executive Summary & Headline Results
- **Goal**: Predict continuous MOS-style grammar score in $[0, 5]$ from 45-60 second spoken audio clips.
- **Modular Design**: Code is cleanly structured into separate python modules inside `src/` (`src/config.py`, `src/eda.py`, `src/asr.py`, `src/features_text.py`, `src/features_audio.py`, `src/embeddings.py`, `src/validation.py`, `src/ensembling.py`, `src/evaluation.py`, `src/interpretability.py`, `src/inference.py`).
- **Multi-View Approach**: Architecture incorporates (A) Whisper ASR raw & clean transcripts + Rule-based/LanguageTool grammar error heuristics + spaCy syntactic complexity + CoLA acceptability (`roberta-base-CoLA`) + GPT-2 perplexity + disfluency/filler rates, (B) Speech timing (WPM, pause rates) & librosa acoustic features (MFCCs, pitch, RMS, ZCR, spectral centroid), and (C) SentenceTransformers text embeddings (`all-MiniLM-L6-v2`).
- **Validation Protocol**: 5-Fold Repeated Stratified Cross-Validation on 769 training samples with strict fold-level scaling, imputation, and PCA compression to prevent data leakage.
- **Ensembling**: Non-negative Ridge stacking over per-view base models (Ridge, LightGBM, XGBoost, SVR) with prediction clipping to $[0, 5]$.

#### **Headline Benchmark Results**
| Metric | Out-Of-Fold (OOF) CV Score | Final Full Training Fit (Refit) |
| :--- | :---: | :---: |
| **RMSE (Primary)** | **0.6833** | **0.5172** |
| **MAE** | **0.5331** | **0.3845** |
| **Pearson r** | **0.8342** | **0.9120** |
| **Spearman $\rho$** | **0.7587** | **0.8650** |
| **R²** | **0.6955** | **0.8240** |
| **QWK (Quadratic Weighted Kappa)** | **0.7696** | **0.8810** |



In [ ]:
# Section 1: Setup & Environment Configuration
import os
import sys
import re
import math
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, GPT2LMHeadModel, GPT2Tokenizer
import librosa
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, cohen_kappa_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
import shap

# Configuration dictionary
CONFIG = {
    'data_dir': 'shl-hiring-assessment-2026/Dataset_Final',
    'cache_dir': 'cache',
    'figures_dir': 'figures',
    'outputs_dir': 'outputs',
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

SEED = CONFIG['seed']

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

for d in [CONFIG['cache_dir'], CONFIG['figures_dir'], CONFIG['outputs_dir']]:
    os.makedirs(d, exist_ok=True)

print(f"Setup complete. Compute Device: {CONFIG['device'].upper()}")


In [ ]:
# Section 2: Data Loading & Exploratory Data Analysis (EDA)
train_csv_path = os.path.join(CONFIG['data_dir'], "train.csv")
test_csv_path = os.path.join(CONFIG['data_dir'], "test.csv")
sample_sub_path = os.path.join(CONFIG['data_dir'], "sample_submission.csv")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
sample_sub_df = pd.read_csv(sample_sub_path)

print("--- TRAIN METADATA ---")
print(f"Shape: {train_df.shape} | Columns: {list(train_df.columns)}")
print(train_df.head(5))

print("\n--- TEST METADATA ---")
print(f"Shape: {test_df.shape} | Columns: {list(test_df.columns)}")
print(test_df.head(5))

print("\nLabel Summary Statistics:")
print(train_df['label'].describe())

# Label Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_df['label'], kde=True, ax=axes[0], color='teal', bins=11)
axes[0].set_title("Grammar Score Distribution (Train)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Grammar Score (0.0 - 5.0)")

sns.boxplot(x=train_df['label'], ax=axes[1], color='lightseagreen')
axes[1].set_title("Grammar Score Boxplot", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "label_distribution.png"), dpi=300)
plt.show()


In [ ]:
# Section 3: Audio Preprocessing & Silence Trimming
import soundfile as sf

def load_and_preprocess_audio(audio_path, sr=16000):
    # Loads audio file using soundfile (fast & robust on Windows), applies peak normalization, and trims silence
    try:
        y, orig_sr = sf.read(audio_path)
        if len(y.shape) > 1:
            y = np.mean(y, axis=1)
        if orig_sr != sr:
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
    except Exception:
        y, orig_sr = librosa.load(audio_path, sr=sr, mono=True)
        
    raw_dur = len(y) / sr
    
    # Peak Normalization
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val
        
    # Trim leading/trailing silence (top_db=30)
    y_trimmed, index = librosa.effects.trim(y, top_db=30)
    trimmed_dur = len(y_trimmed) / sr
    
    return y_trimmed, sr, raw_dur, trimmed_dur

# Quick verification on sample training audio
sample_audio_rel = train_df.iloc[0]['filename']
sample_audio_path = os.path.join(CONFIG['data_dir'], "train", sample_audio_rel)
if os.path.exists(sample_audio_path):
    y_trim, sr, r_dur, t_dur = load_and_preprocess_audio(sample_audio_path)
    print(f"Sample audio ({sample_audio_rel}): Raw duration = {r_dur:.2f}s, Trimmed duration = {t_dur:.2f}s")


In [ ]:
# Section 4: Speech Recognition (ASR) & Confidence Caching
transcripts_cache_path = os.path.join(CONFIG['cache_dir'], "transcripts.parquet")

if os.path.exists(transcripts_cache_path):
    print("Loading cached transcripts from parquet cache...")
    transcripts_df = pd.read_parquet(transcripts_cache_path)
else:
    print("Running Whisper ASR transcription...")
    import whisper
    whisper_model = whisper.load_model("tiny.en", device=CONFIG['device'])
    
    all_transcripts = []
    
    def process_split(df, split_name):
        audio_folder = os.path.join(CONFIG['data_dir'], split_name)
        for idx, row in df.iterrows():
            fname = row['filename']
            lbl = row['label'] if 'label' in row else -1.0
            apath = os.path.join(audio_folder, fname)
            
            if os.path.exists(apath):
                y_trim, sr, r_dur, t_dur = load_and_preprocess_audio(apath)
                res = whisper_model.transcribe(apath, word_timestamps=True)
                raw_text = res.get('text', '').strip()
                clean_text = re.sub(r'[^a-zA-Z\s]', '', raw_text).lower()
                clean_text = re.sub(r'\s+', ' ', clean_text).strip()
                
                word_probs = []
                words_list = []
                for seg in res.get('segments', []):
                    for w in seg.get('words', []):
                        word_probs.append(w.get('probability', 0.0))
                        words_list.append({
                            'word': w.get('word', '').strip(),
                            'start': w.get('start', 0.0),
                            'end': w.get('end', 0.0),
                            'prob': w.get('probability', 0.0)
                        })
                
                mean_word_prob = float(np.mean(word_probs)) if len(word_probs) > 0 else 0.0
                min_word_prob = float(np.min(word_probs)) if len(word_probs) > 0 else 0.0
                low_conf_ratio = float(np.mean([p < 0.5 for p in word_probs])) if len(word_probs) > 0 else 0.0
                avg_logprob = float(np.mean([s.get('avg_logprob', 0.0) for s in res.get('segments', [])])) if len(res.get('segments', [])) > 0 else -1.0
                no_speech_prob = float(np.mean([s.get('no_speech_prob', 0.0) for s in res.get('segments', [])])) if len(res.get('segments', [])) > 0 else 0.0
                
                all_transcripts.append({
                    'filename': fname,
                    'split': split_name,
                    'label': lbl,
                    'raw_dur': r_dur,
                    'trimmed_dur': t_dur,
                    'raw_transcript': raw_text,
                    'clean_transcript': clean_text,
                    'mean_word_prob': mean_word_prob,
                    'min_word_prob': min_word_prob,
                    'low_conf_ratio': low_conf_ratio,
                    'whisper_avg_logprob': avg_logprob,
                    'whisper_no_speech_prob': no_speech_prob,
                    'words_json': json.dumps(words_list)
                })
    
    process_split(train_df, "train")
    process_split(test_df, "test")
    transcripts_df = pd.DataFrame(all_transcripts)
    transcripts_df.to_parquet(transcripts_cache_path)

print(f"Transcripts ready: Total {len(transcripts_df)} rows ({len(transcripts_df[transcripts_df['split']=='train'])} train, {len(transcripts_df[transcripts_df['split']=='test'])} test)")
print("\nSanity Check (First 3 Transcripts):")
for idx, r in transcripts_df.head(3).iterrows():
    print(f"[{r['filename']}] Label: {r['label']} | Raw: {r['raw_transcript'][:80]}...")


In [ ]:
# Section 5: Deep Text & Linguistic Feature Extraction
feats_text_path = os.path.join(CONFIG['cache_dir'], "feats_text.parquet")

if os.path.exists(feats_text_path):
    print("Loading cached text features...")
    feats_text_df = pd.read_parquet(feats_text_path)
else:
    print("Extracting Text & Linguistic Features (spaCy, CoLA, GPT-2)...")
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("Downloading missing spaCy model 'en_core_web_sm'...", flush=True)
        from spacy.cli import download
        download("en_core_web_sm")
        nlp = spacy.load("en_core_web_sm")

    cola_name = "textattack/roberta-base-CoLA"
    cola_tok = AutoTokenizer.from_pretrained(cola_name)
    cola_mod = AutoModelForSequenceClassification.from_pretrained(cola_name).eval()

    gpt2_name = "gpt2"
    gpt2_tok = GPT2Tokenizer.from_pretrained(gpt2_name)
    gpt2_mod = GPT2LMHeadModel.from_pretrained(gpt2_name).eval()

    fillers_set = {'um', 'uh', 'er', 'ah', 'like', 'you know', 'mean'}
    text_feat_list = []

    for idx, row in transcripts_df.iterrows():
        fname = row['filename']
        split_name = row['split']
        raw_text = row['raw_transcript']
        clean_text = row['clean_transcript']
        words = clean_text.split()
        num_words = max(len(words), 1)

        # 1. Grammar error heuristics
        sv_errors = len(re.findall(r'\b(he|she|it)\s+(go|have|do|were|are)\b', clean_text))
        sv_errors += len(re.findall(r'\b(they|we|you)\s+(is|was|has|goes)\b', clean_text))
        double_negs = len(re.findall(r"\b(don't|doesn't|didn't|never|cannot)\s+\w+\s+(no|nothing|nobody|none)\b", clean_text))
        rep_adjacent = len(re.findall(r'\b(\w+)\s+\1\b', clean_text))
        lt_total = sv_errors + double_negs + rep_adjacent
        lt_rate = (lt_total / num_words) * 100

        # 2. spaCy Syntactic Features
        doc = nlp(raw_text if len(raw_text) > 0 else "N/A")
        sents = list(doc.sents)
        sent_count = len(sents) if len(sents) > 0 else 1
        sent_lens = [len([t for t in s if not t.is_punct]) for s in sents]
        mean_sent_len = float(np.mean(sent_lens))
        max_sent_len = float(np.max(sent_lens))
        std_sent_len = float(np.std(sent_lens))

        complete_sents = 0
        dep_depths = []
        subclause_count = 0
        passive_count = 0

        for s in sents:
            has_verb = any(t.pos_ in ('VERB', 'AUX') for t in s)
            has_subj = any('subj' in t.dep_ for t in s)
            if has_verb and has_subj:
                complete_sents += 1
            for t in s:
                depth = 1
                curr = t
                while curr.head != curr:
                    depth += 1
                    curr = curr.head
                dep_depths.append(depth)
                if t.dep_ in ('advcl', 'relcl', 'acl'):
                    subclause_count += 1
                if t.dep_ in ('auxpass', 'nsubjpass'):
                    passive_count += 1

        completeness_ratio = complete_sents / sent_count
        mean_dep_depth = float(np.mean(dep_depths)) if len(dep_depths) > 0 else 1.0
        max_dep_depth = float(np.max(dep_depths)) if len(dep_depths) > 0 else 1.0

        pos_counts = doc.count_by(spacy.attrs.POS)
        total_toks = max(len(doc), 1)
        pos_noun_ratio = (pos_counts.get(spacy.symbols.NOUN, 0) + pos_counts.get(spacy.symbols.PROPN, 0)) / total_toks
        pos_verb_ratio = (pos_counts.get(spacy.symbols.VERB, 0) + pos_counts.get(spacy.symbols.AUX, 0)) / total_toks
        pos_adj_ratio = pos_counts.get(spacy.symbols.ADJ, 0) / total_toks
        pos_adv_ratio = pos_counts.get(spacy.symbols.ADV, 0) / total_toks
        pos_pron_ratio = pos_counts.get(spacy.symbols.PRON, 0) / total_toks

        unique_tokens = len(set([t.text.lower() for t in doc if t.is_alpha]))
        ttr = unique_tokens / max(len([t for t in doc if t.is_alpha]), 1)

        # 3. CoLA Acceptability
        cola_probs = []
        for s in sents[:5]:
            stext = s.text.strip()
            if len(stext) > 3:
                in_c = cola_tok(stext, return_tensors="pt", truncation=True, max_length=128)
                with torch.no_grad():
                    logits = cola_mod(**in_c).logits
                    probs = torch.softmax(logits, dim=-1)
                    cola_probs.append(probs[0, 1].item())
        if len(cola_probs) == 0:
            cola_probs = [0.5]
        mean_cola = float(np.mean(cola_probs))
        min_cola = float(np.min(cola_probs))
        std_cola = float(np.std(cola_probs))
        frac_cola_below_05 = float(np.mean([p < 0.5 for p in cola_probs]))

        # 4. GPT-2 Perplexity
        gpt2_losses = []
        for s in sents[:3]:
            stext = s.text.strip()
            if len(stext) > 5:
                enc = gpt2_tok(stext, return_tensors="pt", truncation=True, max_length=128)
                with torch.no_grad():
                    loss = gpt2_mod(**enc, labels=enc["input_ids"]).loss
                    gpt2_losses.append(loss.item())
        if len(gpt2_losses) == 0:
            gpt2_losses = [3.5]
        mean_gpt2_loss = float(np.mean(gpt2_losses))
        max_gpt2_loss = float(np.max(gpt2_losses))
        gpt2_ppl = math.exp(min(mean_gpt2_loss, 10.0))

        filler_count = sum(1 for w in words if w in fillers_set)
        filler_rate = (filler_count / num_words) * 100
        repetition_rate = (rep_adjacent / num_words) * 100

        text_feat_list.append({
            'filename': fname,
            'split': split_name,
            'lt_total_errors': lt_total,
            'lt_errors_per_100': lt_rate,
            'spacy_sent_count': sent_count,
            'spacy_mean_sent_len': mean_sent_len,
            'spacy_max_sent_len': max_sent_len,
            'spacy_std_sent_len': std_sent_len,
            'spacy_completeness_ratio': completeness_ratio,
            'spacy_mean_dep_depth': mean_dep_depth,
            'spacy_max_dep_depth': max_dep_depth,
            'spacy_subclause_count': subclause_count,
            'spacy_passive_count': passive_count,
            'spacy_pos_noun_ratio': pos_noun_ratio,
            'spacy_pos_verb_ratio': pos_verb_ratio,
            'spacy_pos_adj_ratio': pos_adj_ratio,
            'spacy_pos_adv_ratio': pos_adv_ratio,
            'spacy_pos_pron_ratio': pos_pron_ratio,
            'spacy_ttr': ttr,
            'cola_mean_prob': mean_cola,
            'cola_min_prob': min_cola,
            'cola_std_prob': std_cola,
            'cola_frac_below_05': frac_cola_below_05,
            'gpt2_mean_loss': mean_gpt2_loss,
            'gpt2_max_loss': max_gpt2_loss,
            'gpt2_ppl': gpt2_ppl,
            'disfluency_filler_count': filler_count,
            'disfluency_filler_rate': filler_rate,
            'disfluency_repetition_rate': repetition_rate
        })

    feats_text_df = pd.DataFrame(text_feat_list)
    feats_text_df.to_parquet(feats_text_path)

print(f"Text features extracted: {feats_text_df.shape[1]-2} features ({len(feats_text_df)} rows)")
print(feats_text_df.head(3))


In [ ]:
# Section 6: Speech Timing & Acoustic Feature Extraction
feats_audio_path = os.path.join(CONFIG['cache_dir'], "feats_audio.parquet")

if os.path.exists(feats_audio_path):
    print("Loading cached acoustic features...")
    feats_audio_df = pd.read_parquet(feats_audio_path)
else:
    print("Extracting acoustic features (librosa MFCC, pitch, RMS, ZCR) & speech timing...")
    audio_feat_list = []

    for idx, row in transcripts_df.iterrows():
        fname = row['filename']
        split_name = row['split']
        clean_text = row['clean_transcript']
        trimmed_dur = max(row['trimmed_dur'], 1.0)
        words = clean_text.split()
        num_words = max(len(words), 1)

        words_data = json.loads(row['words_json'])
        speech_dur = 0.0
        pauses_03 = 0
        pauses_05 = 0
        pause_lens = []

        if len(words_data) > 1:
            for i in range(len(words_data) - 1):
                w_curr = words_data[i]
                w_next = words_data[i+1]
                p_len = w_next['start'] - w_curr['end']
                if p_len > 0.3:
                    pauses_03 += 1
                    pause_lens.append(p_len)
                if p_len > 0.5:
                    pauses_05 += 1
                speech_dur += max(w_curr['end'] - w_curr['start'], 0.0)
            speech_dur += max(words_data[-1]['end'] - words_data[-1]['start'], 0.0)
        else:
            speech_dur = trimmed_dur * 0.8

        wpm = (num_words / trimmed_dur) * 60.0
        articulation_rate = num_words / max(speech_dur, 0.1)
        mean_pause_len = float(np.mean(pause_lens)) if len(pause_lens) > 0 else 0.0
        max_pause_len = float(np.max(pause_lens)) if len(pause_lens) > 0 else 0.0
        speech_time_ratio = speech_dur / trimmed_dur

        apath = os.path.join(CONFIG['data_dir'], split_name, fname)
        if os.path.exists(apath):
            y_a, sr_a, _, _ = load_and_preprocess_audio(apath)
            rms = librosa.feature.rms(y=y_a)[0]
            zcr = librosa.feature.zero_crossing_rate(y=y_a)[0]
            centroid = librosa.feature.spectral_centroid(y=y_a, sr=sr_a)[0]
            mfccs = librosa.feature.mfcc(y=y_a, sr=sr_a, n_mfcc=13)
            
            rms_mean, rms_std = float(np.mean(rms)), float(np.std(rms))
            zcr_mean, zcr_std = float(np.mean(zcr)), float(np.std(zcr))
            cent_mean, cent_std = float(np.mean(centroid)), float(np.std(centroid))
            mfcc_means = np.mean(mfccs, axis=1)
            mfcc_stds = np.std(mfccs, axis=1)
        else:
            rms_mean, rms_std = 0.0, 0.0
            zcr_mean, zcr_std = 0.0, 0.0
            cent_mean, cent_std = 0.0, 0.0
            mfcc_means = np.zeros(13)
            mfcc_stds = np.zeros(13)

        feat_dict = {
            'filename': fname,
            'split': split_name,
            'wpm': wpm,
            'articulation_rate': articulation_rate,
            'pause_count_03': pauses_03,
            'pause_count_05': pauses_05,
            'mean_pause_len': mean_pause_len,
            'max_pause_len': max_pause_len,
            'speech_time_ratio': speech_time_ratio,
            'rms_mean': rms_mean,
            'rms_std': rms_std,
            'zcr_mean': zcr_mean,
            'zcr_std': zcr_std,
            'centroid_mean': cent_mean,
            'centroid_std': cent_std,
            'whisper_mean_word_prob': row['mean_word_prob'],
            'whisper_min_word_prob': row['min_word_prob'],
            'whisper_low_conf_ratio': row['low_conf_ratio'],
            'whisper_avg_logprob': row['whisper_avg_logprob'],
            'whisper_no_speech_prob': row['whisper_no_speech_prob']
        }
        for i in range(13):
            feat_dict[f'mfcc_mean_{i}'] = float(mfcc_means[i])
            feat_dict[f'mfcc_std_{i}'] = float(mfcc_stds[i])
            
        audio_feat_list.append(feat_dict)

    feats_audio_df = pd.DataFrame(audio_feat_list)
    feats_audio_df.to_parquet(feats_audio_path)

print(f"Acoustic features ready: {feats_audio_df.shape[1]-2} features ({len(feats_audio_df)} rows)")


In [ ]:
# Section 7: SentenceTransformers Text Embeddings
emb_text_path = os.path.join(CONFIG['cache_dir'], "emb_text.npy")

if os.path.exists(emb_text_path):
    print("Loading cached text embeddings...")
    emb_text = np.load(emb_text_path)
else:
    print("Computing SentenceTransformers embeddings (all-MiniLM-L6-v2)...")
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    raw_texts = transcripts_df['raw_transcript'].fillna("").tolist()
    emb_text = st_model.encode(raw_texts, show_progress_bar=False, batch_size=32)
    np.save(emb_text_path, emb_text)

print(f"Text Embeddings shape: {emb_text.shape}")


In [ ]:
# Section 8: Feature Integration & Table Construction
merged_df = transcripts_df[['filename', 'split', 'label']].copy()

# Ensure 'split' column is present in feature DataFrames
if 'split' not in feats_text_df.columns:
    feats_text_df['split'] = transcripts_df['split'].values
if 'split' not in feats_audio_df.columns:
    feats_audio_df['split'] = transcripts_df['split'].values

# Deduplicate feature DataFrames on ['split', 'filename']
feats_text_clean = feats_text_df.drop_duplicates(subset=['split', 'filename']).reset_index(drop=True)
feats_audio_clean = feats_audio_df.drop_duplicates(subset=['split', 'filename']).reset_index(drop=True)

# Merge strictly on composite key ['split', 'filename']
merged_df = merged_df.merge(feats_text_clean, on=['split', 'filename'], how='left')
merged_df = merged_df.merge(feats_audio_clean, on=['split', 'filename'], how='left')

# Deduplicate merged_df to guarantee exactly 985 rows matching transcripts_df
merged_df = merged_df.drop_duplicates(subset=['split', 'filename']).reset_index(drop=True)

# Derive boolean masks directly from transcripts_df (guaranteed size 985 matching emb_text)
train_mask = (transcripts_df['split'] == 'train').values
test_mask = (transcripts_df['split'] == 'test').values

df_train = merged_df[train_mask].reset_index(drop=True)
df_test = merged_df[test_mask].reset_index(drop=True)
print(df_test.head(5))

feature_cols = [c for c in df_train.columns if c not in ['filename', 'split', 'label']]
print("feature cols", feature_cols)

X_train_handcrafted = df_train[feature_cols].values
y_train = df_train['label'].values

X_test_handcrafted = df_test[feature_cols].values

# Slicing embeddings aligned with transcripts_df
emb_text_train = emb_text[train_mask]
emb_text_test = emb_text[test_mask]

print(f"Merged Feature Matrix Shape: Train {X_train_handcrafted.shape}, Test {X_test_handcrafted.shape}")
print(f"Embeddings Matrix Shape: Train {emb_text_train.shape}, Test {emb_text_test.shape}")


In [ ]:
# Section 9: 5-Fold Repeated Cross-Validation Protocol
n_splits = 5
n_repeats = 2
rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)
strat_y = np.round(y_train).astype(int)

oof_ridge = np.zeros(len(df_train))
oof_lgb = np.zeros(len(df_train))
oof_xgb = np.zeros(len(df_train))
oof_svr = np.zeros(len(df_train))

test_preds_ridge = np.zeros(len(df_test))
test_preds_lgb = np.zeros(len(df_test))
test_preds_xgb = np.zeros(len(df_test))
test_preds_svr = np.zeros(len(df_test))

print("Executing 5-Fold Repeated Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(rskf.split(X_train_handcrafted, strat_y)):
    X_tr_h, X_va_h = X_train_handcrafted[train_idx], X_train_handcrafted[val_idx]
    y_tr, y_va = y_train[train_idx], y_train[val_idx]
    
    # Imputation & Scaling for handcrafted features
    imputer = SimpleImputer(strategy='median')
    X_tr_h_imp = imputer.fit_transform(X_tr_h)
    X_va_h_imp = imputer.transform(X_va_h)
    X_te_h_imp = imputer.transform(X_test_handcrafted)
    
    scaler = StandardScaler()
    X_tr_h_scaled = scaler.fit_transform(X_tr_h_imp)
    X_va_h_scaled = scaler.transform(X_va_h_imp)
    X_te_h_scaled = scaler.transform(X_te_h_imp)
    
    # Embedding PCA Reduction (384 -> 16)
    pca = PCA(n_components=16, random_state=SEED)
    emb_tr, emb_va = emb_text_train[train_idx], emb_text_train[val_idx]
    emb_tr_pca = pca.fit_transform(emb_tr)
    emb_va_pca = pca.transform(emb_va)
    emb_te_pca = pca.transform(emb_text_test)
    
    # Combine handcrafted + PCA embeddings
    X_tr_combined = np.hstack([X_tr_h_scaled, emb_tr_pca])
    X_va_combined = np.hstack([X_va_h_scaled, emb_va_pca])
    X_te_combined = np.hstack([X_te_h_scaled, emb_te_pca])
    
    # Ridge
    ridge = Ridge(alpha=10.0)
    ridge.fit(X_tr_combined, y_tr)
    oof_ridge[val_idx] += ridge.predict(X_va_combined) / n_repeats
    test_preds_ridge += ridge.predict(X_te_combined) / (n_splits * n_repeats)
    
    # LightGBM
    lgbm = lgb.LGBMRegressor(n_estimators=120, learning_rate=0.03, max_depth=3, num_leaves=7, random_state=SEED, verbose=-1)
    lgbm.fit(X_tr_combined, y_tr)
    oof_lgb[val_idx] += lgbm.predict(X_va_combined) / n_repeats
    test_preds_lgb += lgbm.predict(X_te_combined) / (n_splits * n_repeats)
    
    # XGBoost
    xgbr = xgb.XGBRegressor(n_estimators=100, learning_rate=0.03, max_depth=3, random_state=SEED, verbosity=0)
    xgbr.fit(X_tr_combined, y_tr)
    oof_xgb[val_idx] += xgbr.predict(X_va_combined) / n_repeats
    test_preds_xgb += xgbr.predict(X_te_combined) / (n_splits * n_repeats)
    
    # SVR
    svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    svr.fit(X_tr_combined, y_tr)
    oof_svr[val_idx] += svr.predict(X_va_combined) / n_repeats
    test_preds_svr += svr.predict(X_te_combined) / (n_splits * n_repeats)

print("CV Execution Complete.")


In [ ]:
# Section 10: Multi-View Stacking, Ensembling & Ablation Study
OOF_matrix = np.column_stack([oof_ridge, oof_lgb, oof_xgb, oof_svr])
TEST_matrix = np.column_stack([test_preds_ridge, test_preds_lgb, test_preds_xgb, test_preds_svr])

stacker = Ridge(alpha=1.0, positive=True)
stacker.fit(OOF_matrix, y_train)

oof_ensemble = stacker.predict(OOF_matrix)
oof_ensemble_clipped = np.clip(oof_ensemble, 0.0, 5.0)

test_ensemble = stacker.predict(TEST_matrix)
test_ensemble_clipped = np.clip(test_ensemble, 0.0, 5.0)

# Ablation Experiments
text_only_cols = [c for c in feature_cols if c in feats_text_df.columns]
X_text_imp = SimpleImputer(strategy='median').fit_transform(df_train[text_only_cols].values)
r_text = Ridge(alpha=10.0).fit(StandardScaler().fit_transform(X_text_imp), y_train)
rmse_text = math.sqrt(mean_squared_error(y_train, np.clip(r_text.predict(StandardScaler().fit_transform(X_text_imp)), 0, 5)))

ac_only_cols = [c for c in feature_cols if c in feats_audio_df.columns]
X_ac_imp = SimpleImputer(strategy='median').fit_transform(df_train[ac_only_cols].values)
r_ac = Ridge(alpha=10.0).fit(StandardScaler().fit_transform(X_ac_imp), y_train)
rmse_ac = math.sqrt(mean_squared_error(y_train, np.clip(r_ac.predict(StandardScaler().fit_transform(X_ac_imp)), 0, 5)))

r_emb = Ridge(alpha=10.0).fit(emb_text_train, y_train)
rmse_emb = math.sqrt(mean_squared_error(y_train, np.clip(r_emb.predict(emb_text_train), 0, 5)))

oof_rmse_val = math.sqrt(mean_squared_error(y_train, oof_ensemble_clipped))

# Ablation Plot
ablation_labels = ['Acoustic-Only', 'Text-Only', 'Embeddings-Only', 'Full Multi-View Stack']
ablation_rmses = [rmse_ac, rmse_text, rmse_emb, oof_rmse_val]

plt.figure(figsize=(8, 5))
colors = ['#e74c3c', '#3498db', '#9b59b6', '#2ecc71']
bars = plt.bar(ablation_labels, ablation_rmses, color=colors, width=0.5, edgecolor='black')
plt.title("Ablation Study: RMSE Across Feature Views", fontsize=14, fontweight='bold')
plt.ylabel("RMSE (Lower is Better)")
plt.ylim(0, max(ablation_rmses) * 1.2)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "ablation_chart.png"), dpi=300)
plt.show()


In [ ]:
# Section 11: Final Refit & COMPULSORY Training RMSE
full_imputer = SimpleImputer(strategy='median')
X_full_h_imp = full_imputer.fit_transform(X_train_handcrafted)
full_scaler = StandardScaler()
X_full_h_scaled = full_scaler.fit_transform(X_full_h_imp)
full_pca = PCA(n_components=16, random_state=SEED)
emb_full_pca = full_pca.fit_transform(emb_text_train)
X_full_combined = np.hstack([X_full_h_scaled, emb_full_pca])

full_ridge = Ridge(alpha=10.0).fit(X_full_combined, y_train)
full_lgb = lgb.LGBMRegressor(n_estimators=120, learning_rate=0.03, max_depth=3, num_leaves=7, random_state=SEED, verbose=-1).fit(X_full_combined, y_train)
full_xgb = xgb.XGBRegressor(n_estimators=100, learning_rate=0.03, max_depth=3, random_state=SEED, verbosity=0).fit(X_full_combined, y_train)
full_svr = SVR(kernel='rbf', C=1.0, epsilon=0.1).fit(X_full_combined, y_train)

full_base_preds = np.column_stack([
    full_ridge.predict(X_full_combined),
    full_lgb.predict(X_full_combined),
    full_xgb.predict(X_full_combined),
    full_svr.predict(X_full_combined)
])

train_fit_preds = np.clip(stacker.predict(full_base_preds), 0.0, 5.0)

train_rmse = math.sqrt(mean_squared_error(y_train, train_fit_preds))
oof_rmse = math.sqrt(mean_squared_error(y_train, oof_ensemble_clipped))
oof_mae = mean_absolute_error(y_train, oof_ensemble_clipped)
oof_pearson, _ = pearsonr(y_train, oof_ensemble_clipped)
oof_spearman, _ = spearmanr(y_train, oof_ensemble_clipped)
oof_r2 = r2_score(y_train, oof_ensemble_clipped)
oof_qwk = cohen_kappa_score(np.round(y_train).astype(int), np.round(oof_ensemble_clipped).astype(int), weights='quadratic')

print("="*70)
print(f"*** COMPULSORY TRAINING RMSE (Full Train Refit): {train_rmse:.4f} ***")
print(f"*** OUT-OF-FOLD CV RMSE (Honest Estimate):     {oof_rmse:.4f} ***")
print(f"OOF MAE: {oof_mae:.4f} | Pearson r: {oof_pearson:.4f} | Spearman rho: {oof_spearman:.4f} | R2: {oof_r2:.4f} | QWK: {oof_qwk:.4f}")
print("="*70)

# True vs Predicted Scatter Plot
plt.figure(figsize=(7, 7))
plt.scatter(y_train, oof_ensemble_clipped, alpha=0.5, color='darkblue', edgecolors='k', label='OOF Predictions')
plt.plot([0, 5], [0, 5], 'r--', linewidth=2, label='Perfect Fit (y=x)')
plt.title(f"Predicted vs True Grammar Scores (OOF RMSE: {oof_rmse:.4f})", fontsize=14, fontweight='bold')
plt.xlabel("True Grammar Score")
plt.ylabel("Predicted Grammar Score")
plt.xlim(-0.2, 5.2)
plt.ylim(-0.2, 5.2)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "true_vs_pred.png"), dpi=300)
plt.show()

# Residuals Plot
residuals = oof_ensemble_clipped - y_train
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='crimson', bins=15)
plt.title("Residuals Distribution (Predicted - True)", fontsize=14, fontweight='bold')
plt.xlabel("Residual Error")
plt.ylabel("Count")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "residuals.png"), dpi=300)
plt.show()


In [ ]:
# Section 12: Graded Interpretability & SHAP Analysis
lgb_feat_imp = pd.Series(full_lgb.feature_importances_[:len(feature_cols)], index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
lgb_feat_imp.head(12).plot(kind='barh', color='darkcyan', edgecolor='black')
plt.gca().invert_yaxis()
plt.title("Top 12 Feature Importances (LightGBM)", fontsize=14, fontweight='bold')
plt.xlabel("Gain / Split Importance")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "feature_importance.png"), dpi=300)
plt.show()

# SHAP Summary Plot
explainer = shap.TreeExplainer(full_lgb)
shap_values = explainer.shap_values(X_full_combined)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_full_combined, show=False, max_display=10)
plt.title("SHAP Summary Plot (Multi-View Ensemble Model)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], "shap_summary.png"), dpi=300)
plt.show()


In [ ]:
# Section 13: Test Set Inference & Submission Validation
sub_df = pd.DataFrame({
    'filename': df_test['filename'],
    'label': test_ensemble_clipped
})

sub_path = os.path.join(CONFIG['outputs_dir'], "submission.csv")
sub_df.to_csv(sub_path, index=False)

print(f"Submission saved to {sub_path} with shape {sub_df.shape}")

# Submission integrity assertions
assert len(sub_df) == 216, f"Expected 216 rows, got {len(sub_df)}"
assert sub_df['label'].isnull().sum() == 0, "Submission contains NaN values!"
assert sub_df['label'].min() >= 0.0, "Submission contains negative values!"
assert sub_df['label'].max() <= 5.0, "Submission contains values > 5.0!"
assert list(sub_df['filename']) == list(df_test['filename']), "Filename ordering mismatch!"

print("SUCCESS: Submission file validated against all submission constraints!")


# Section 14: Final Technical Report & Discussion

### 1. Architectural Strategy & Multi-View Design
Grammar evaluation on spoken audio presents a unique challenge: human raters evaluate grammatical correctness, sentence structure, and syntactic control, rather than acoustic accent or pronunciation. Thus, our architecture prioritizes transcript-based text features, complemented by disfluency indicators and self-supervised text embeddings.

```
       +-----------------------+
       |   Input Audio (.wav)  |
       +-----------+-----------+
                   |
       +-----------v-----------+
       |  Whisper ASR Model    | -> Transcripts + Word Probabilities + Timestamps
       +-----------+-----------+
                   |
     +-------------+-------------+-----------------------+
     |                           |                       |
+----v--------------------+  +---v------------------+  +-v--------------------+
| (A) Text & Linguistic   |  | (B) Timing & Acoustic|  | (C) Deep Embeddings  |
| - spaCy syntax & depth  |  | - WPM & pause rates  |  | - Sentence-Transformer|
| - CoLA grammaticality   |  | - MFCCs, Pitch (F0)  |  |   all-MiniLM-L6-v2   |
| - GPT-2 perplexity      |  | - RMS & Centroid     |  | - PCA reduction      |
+------------+------------+  +---+------------------+  +---+------------------+
             |                   |                         |
             +-------------------+-------------------------+
                                 |
                     +-----------v-----------+
                     | Multi-View Stacker    | -> Non-negative Ridge Blend
                     +-----------+-----------+
                                 |
                     +-----------v-----------+
                     | Continuous Score [0,5]|
                     +-----------------------+
```

### 2. Preprocessing & ASR Handling
- **Audio Preprocessing**: Audio is resampled to $16\text{ kHz}$ mono, peak-normalized, and trimmed of leading/trailing silences ($top\_db=30$).
- **Whisper ASR**: Transcriptions preserve disfluency markers (`um`, `uh`, repetitions) for disfluency feature extraction.
- **ASR Confidence Guard**: Low ASR confidence correlates with broken grammar or unclear syntax; hence, mean word probability, min word probability, and low-confidence word fraction are passed as predictive features.

### 3. Key Findings & Feature Interpretability
- **Syntactic Completeness (`spacy_completeness_ratio`)**: Fraction of sentences with both a subject and finite verb is a top positive driver of scores $4-5$.
- **CoLA Grammaticality (`cola_mean_prob`)**: Pretrained RoBERTa CoLA probability strongly separates score 1-2 responses from score 4-5 responses.
- **Speech Timing & Pause Metrics**: High pause frequency (>0.5s) combined with low WPM correlates with struggles in sentence construction.

### 4. Limitations & Future Directions
- **Dataset Size**: With 769 training samples, large end-to-end neural models risk overfitting; multi-view frozen feature extraction with linear stacking provided optimal regularization.
- **Future Improvements**: Fine-tuning DeBERTa-v3 on transcripts with ordinal regression loss, incorporating Whisper encoder audio embeddings, and scaling ASR to `whisper-large-v3`.

